#### Hybrid Simulation Using PySD and Parallel Programming

In [10]:
import psutil
import pysd
import pandas as pd
from joblib import Parallel, delayed

In [11]:
# identify the number of physical cores
print("Number of Physical Cores: ", psutil.cpu_count(logical=False))

Number of Physical Cores:  4


In [21]:
# 1. PRE-TRANSLATE (Done once by the main process)
# This creates the .py file version of your model so workers don't have to
model_filename = "Glossi v2.mdl"
py_model_file = pysd.read_vensim(model_filename)
# Get the path of the generated python file
py_path = py_model_file.py_model_file

#### Define a parallelable function

In [22]:
def run_single_simulation(seed) -> pd.DataFrame:
    """
    Each CPU core will now execute this function, load its own 
    copy of the model, run it, and then clear it from memory.
    """
    # Load the model INSIDE the function so it stays on this core
    local_model = pysd.load(py_path)
    
    # Run the simulation with the specific seed
    result = local_model.run(params={'Model Seed': seed})
    
    return result['Dryness Level']

#### Setting number of runs

In [23]:
# set number of samples and seeds
num_samples = 10000
seeds = range(num_samples)

#### Parallelizing the simulation

In [24]:
print(f"Starting {num_samples} parallel runs...")

list_of_results = Parallel(n_jobs=-1)(
    delayed(run_single_simulation)(seed) for seed in seeds
)

Starting 10000 parallel runs...


#### Combine results in a master DataFrame

In [25]:
all_runs_df = pd.concat(list_of_results, axis=1)

print("ALL simulations complete.")
print(all_runs_df.head())

ALL simulations complete.
      Dryness Level  Dryness Level  Dryness Level  Dryness Level  \
time                                                               
0             4.915          4.915          4.915          4.915   
1             4.929          4.495          4.929          4.495   
2             4.509          4.509          4.509          4.509   
3             4.523          4.523          4.523          4.523   
4             4.537          4.103          4.103          4.537   

      Dryness Level  Dryness Level  Dryness Level  Dryness Level  \
time                                                               
0             4.915          4.915          4.915          4.915   
1             4.929          4.495          4.929          4.929   
2             4.943          4.509          4.509          4.943   
3             4.957          4.089          4.089          4.523   
4             4.971          4.103          4.103          4.103   

      Dryness Level 